# Bloco 6 — BVAR com Minnesota prior

Implementação Bayesiana do VAR estimado no Bloco 5, com Minnesota prior e 
calibração do hiperparâmetro de encolhimento via marginal likelihood 
(Giannone, Lenza & Primiceri, 2015).

A fundamentação metodológica completa está documentada em
`docs/metodologia/bvar_minnesota.md`. Este notebook foca na 
**implementação**, com a parte conceitual reduzida ao mínimo necessário 
para acompanhar o código.

## Estrutura do bloco

| Fase | Conteúdo |
|:---:|:---|
| 6.1 | BVAR Minnesota — implementação manual, forma fechada |
| 6.2 | Replicação via PyMC |
| 6.3 | Comparação VAR clássico × BVAR |

## Decisões metodológicas adotadas

- Endógenas: as 6 do VAR(4) baseline do Bloco 5
- Exógenas: 7 dummies de pandemia
- Defasagens: $p=4$ (mesmo do baseline)
- $\delta_i = 0$ para todas (séries em diferença ou I(0))
- $\theta = 0{,}5$, $\alpha = 2$ (convenções de Litterman)
- $\lambda$: calibrado via marginal likelihood
- Identificação: Cholesky (mesma ordem do Bloco 5)
- Horizonte das IRFs: 24 meses

In [4]:
# =============================================================================
# 6.1 BVAR Minnesota — Setup
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

from scipy import linalg
from scipy.special import gammaln, multigammaln
from scipy.stats import invwishart, matrix_normal

import statsmodels.api as sm
from statsmodels.tsa.api import VAR

warnings.filterwarnings('ignore')

# Reprodutibilidade
np.random.seed(42)

# Caminhos
DATA_PATH = Path('../data/processed/series_tratadas.csv')
FIGURES_PATH = Path('../outputs/figures')
TABLES_PATH = Path('../outputs/tables')

# Carrega os dados tratados (saída do Bloco 3)
df = pd.read_csv(DATA_PATH, parse_dates=['Date'], index_col='Date')

print(f"Dados carregados:")
print(f"  Shape: {df.shape}")
print(f"  Período: {df.index.min().date()} a {df.index.max().date()}")
print(f"  Colunas: {list(df.columns)}")

Dados carregados:
  Shape: (276, 16)
  Período: 2003-01-01 a 2025-12-01
  Colunas: ['ln_ibcbr', 'ln_cambio', 'ln_commodities', 'ln_m1', 'ln_prod_industrial', 'ln_credito_total', 'ipca', 'selic', 'exp_ipca_12m', 'dummy_covid_2020_03', 'dummy_covid_2020_04', 'dummy_covid_2020_05', 'dummy_covid_2020_06', 'dummy_covid_2020_07', 'dummy_covid_2020_08', 'dummy_covid_2020_09']


## 2. Construção das matrizes do VAR em forma compacta

O VAR(p) em forma matricial:

$$Y = X B + U$$

onde $Y$ é $(T \times K)$ com observações empilhadas das $K$ endógenas, 
e $X$ é $(T \times Kp + m)$ contendo intercepto, lags das endógenas e 
variáveis exógenas, na ordem:

$$X_t = \begin{bmatrix} 1 & y_{t-1}' & y_{t-2}' & \cdots & y_{t-p}' & x_t' \end{bmatrix}$$

Esta construção segue a convenção do `statsmodels` para o VAR clássico, 
permitindo comparação direta entre OLS (Bloco 5) e Bayesiano (Bloco 6).

In [5]:
# =============================================================================
# 2. Construção das matrizes Y e X
# =============================================================================

# Endógenas (mesma ordem de Cholesky do Bloco 5 — Etapa 5.5)
ENDOG_NAMES = [
    'd_ln_commodities',
    'd_ln_ibcbr',
    'ipca',
    'd_exp_ipca_12m',
    'd_selic',
    'd_ln_cambio',
]

# Exógenas (7 dummies de pandemia)
EXOG_NAMES = [c for c in df.columns if c.startswith('dummy_covid_')]

# Defasagens (mesma especificação do baseline)
P_LAGS = 4

# Extrai os arrays
Y_full = df[ENDOG_NAMES].values  # (T_full, K)
X_exog_full = df[EXOG_NAMES].values  # (T_full, M)

T_full, K = Y_full.shape
_, M = X_exog_full.shape

print(f"Antes de construir lags:")
print(f"  T_full: {T_full}")
print(f"  K (endógenas): {K}")
print(f"  M (exógenas): {M}")
print(f"  p (defasagens): {P_LAGS}")


def construir_matriz_var(Y, X_exog, p):
    """
    Constrói (Y_dep, X_reg) no formato do VAR(p):
        Y_dep[t] = c + A_1 Y[t-1] + ... + A_p Y[t-p] + B X_exog[t] + u[t]
    
    Parâmetros
    ----------
    Y : array (T, K) — endógenas
    X_exog : array (T, M) — exógenas (sem coluna de intercepto)
    p : int — número de defasagens
    
    Retorna
    -------
    Y_dep : array (T-p, K) — variável dependente
    X_reg : array (T-p, 1 + K*p + M) — regressores empilhados
        Ordem das colunas: [intercepto, Y_{t-1}, Y_{t-2}, ..., Y_{t-p}, X_exog_t]
    """
    T, K_ = Y.shape
    _, M_ = X_exog.shape
    
    T_efetivo = T - p
    
    # Variável dependente: observações a partir do tempo p
    Y_dep = Y[p:, :]  # shape (T-p, K)
    
    # Matriz de regressores
    n_cols = 1 + K_ * p + M_  # intercepto + lags + exógenas
    X_reg = np.zeros((T_efetivo, n_cols))
    
    # Coluna 0: intercepto
    X_reg[:, 0] = 1.0
    
    # Lags das endógenas: do mais recente (t-1) ao mais antigo (t-p)
    for lag in range(1, p + 1):
        col_start = 1 + (lag - 1) * K_
        col_end = 1 + lag * K_
        X_reg[:, col_start:col_end] = Y[p - lag : T - lag, :]
    
    # Variáveis exógenas (contemporâneas)
    X_reg[:, 1 + K_ * p:] = X_exog[p:, :]
    
    return Y_dep, X_reg


# Constrói as matrizes
Y_dep, X_reg = construir_matriz_var(Y_full, X_exog_full, P_LAGS)
T = Y_dep.shape[0]
n_coefs = X_reg.shape[1]

print(f"\nApós construção do VAR:")
print(f"  T (observações efetivas): {T}")
print(f"  Y_dep shape: {Y_dep.shape}")
print(f"  X_reg shape: {X_reg.shape}")
print(f"  Coeficientes por equação: {n_coefs}")
print(f"  Total de coeficientes do VAR: {n_coefs * K}")
print(f"\nEstrutura das colunas de X_reg:")
print(f"  Coluna 0:        intercepto")
print(f"  Colunas 1-{K*P_LAGS}:     lags de Y (do mais recente ao mais antigo)")
print(f"  Colunas {1+K*P_LAGS}-{n_coefs-1}:  dummies de pandemia")

KeyError: "['d_ln_commodities', 'd_ln_ibcbr', 'd_exp_ipca_12m', 'd_selic', 'd_ln_cambio'] not in index"